In [ ]:
%cd ../..

In [ ]:
import numpy as np
from csr.module.dataset.rrclarc_datasets import get_celeba_biased_dataset


In [ ]:
celeba_path = ['/media/disk1/Data/',]
artifact_path = 'configs/dataset/artifacts_celeba.json'

dataset = get_celeba_biased_dataset(
    data_paths=celeba_path,
    normalize_data=True,
    artifact_ids_file=artifact_path
)


In [ ]:
print(dataset.sample_ids_by_artifact.keys())
print([len(dataset.sample_ids_by_artifact[artifact_id]) for artifact_id in dataset.sample_ids_by_artifact.keys()])
print()
print(len(dataset.all_artifact_sample_ids))
print(len(dataset.clean_sample_ids))
print(len(dataset))
print()
print(len(set.intersection(set(dataset.all_artifact_sample_ids), set(dataset.idxs_val))))
print(len(set.intersection(set(dataset.all_artifact_sample_ids), set(dataset.idxs_test))))
print(len(set.intersection(set(dataset.all_artifact_sample_ids), set(dataset.idxs_train))))
print(len(set.intersection(set(dataset.clean_sample_ids), set(dataset.idxs_val))))
print(len(set.intersection(set(dataset.clean_sample_ids), set(dataset.idxs_test))))
print(len(set.intersection(set(dataset.clean_sample_ids), set(dataset.idxs_train))))
print()
print(len(np.intersect1d(dataset.all_artifact_sample_ids, dataset.idxs_val)))





In [ ]:
from torchvision.datasets import CelebA
ds = CelebA(
    root = '/media/disk1/Data/',
    split = 'all',
)

In [ ]:
ATTR = "Blond_Hair"
ATTR2 = "Wearing_Necktie"
attr_id = np.where(np.array(ds.attr_names) == ATTR)[0][0]
attr_id2 = np.where(np.array(ds.attr_names) == ATTR2)[0][0]
USE_SUBSET = True
if USE_SUBSET:
    print("Using subset")
    NTH = 10
else:
    NTH = 1
filter_indices = np.zeros(len(ds.attr))


filter_indices[::NTH] = 1
print(f"Chosen attribute {ATTR} with id {attr_id}.")
labels = ds.attr[:, attr_id]
labels2 = ds.attr[:, attr_id2]
both = np.where(np.logical_and(labels, labels2) == 1)[0]
# print(both)
filter_indices[both] = 1

both_names = np.array(ds.filename)[both]
# print("', '".join(both_names) + "\n")

labels_not_blonde = 1 * (ds.attr[:, attr_id] == 0)
labels_collar = ds.attr[:, attr_id2]
not_blonde_collar = np.where(
    np.logical_and(labels_not_blonde, labels_collar) == 1
)[0]
print("not blonde", len(not_blonde_collar))
print("overall", np.sum(filter_indices == 1))
filter_indices[not_blonde_collar] = 1

labels = ds.attr[:, attr_id][filter_indices == 1]
labels2 = ds.attr[:, attr_id2][filter_indices == 1]
both = np.where(np.logical_and(labels, labels2) == 1)[0]
# print(both[:10])

labels_not_blonde = 1 * (ds.attr[:, attr_id][filter_indices == 1] == 0)
labels_collar = ds.attr[:, attr_id2][filter_indices == 1]
not_blonde_collar = np.where(
    np.logical_and(labels_not_blonde, labels_collar) == 1
)[0]

print(f"Chosen attribute {ATTR} with id {attr_id}.")

In [ ]:
dataset.metadata

In [ ]:
dataset.metadata['targets'].value_counts()

In [ ]:
dataset.metadata['blonde_collar'].value_counts()

In [ ]:
dataset.idxs_train

In [ ]:
dset_tr = dataset.get_subset_by_idxs(dataset.idxs_train)

In [ ]:
dset_tr.metadata

In [ ]:
def unnormalize(img):
    return img.permute(1,2,0) * 0.5 + 0.5


In [ ]:
# plot some images in dataset_dict
from itertools import product
import matplotlib.pyplot as plt
import torchvision

fig, ax = plt.subplots(4, 10, figsize=(30, 12))

for tmp in [[0, 0], [0, 1], [1, 0], [1, 1]]:
    targets, blonde_collar = tmp
    row = targets * 2 + blonde_collar
    for col in range(10):
        idxs = dset_tr.metadata.query(f"targets == {targets} and blonde_collar == {blonde_collar}").index
        if len(idxs) == 0:
            continue

        img, label = dset_tr[idxs[col]]
        ax[row, col].imshow(unnormalize(img))
        ax[row, col].set_title(label)
        ax[row, col].axis('off')
plt.show()

---


In [1]:
%cd ../..

/home/jj/Research/ConceptualSensitivityRegularization


In [2]:
from csr.module import DataModule


/home/jj/anaconda3/envs/torch2.1_cuda11.8/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/jj/anaconda3/envs/torch2.1_cuda11.8/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [5]:
dm = DataModule(
    dataset='celeba_collar',
    data_type='feature',
    data_dir='/media/disk2/Data',
    model='convnext_t',
    minor_ratio = 0
)

In [6]:
dm.setup()
dm.prepare_data()

tr {(0, 0): tensor(24235), (0, 1): tensor(2395), (1, 0): tensor(0), (1, 1): tensor(0)}
va {(0, 0): tensor(6090), (0, 1): tensor(570), (1, 0): tensor(0), (1, 1): tensor(174)}
te {(0, 0): tensor(6040), (0, 1): tensor(614), (1, 0): tensor(0), (1, 1): tensor(180)}


In [13]:
dm = DataModule(
    dataset='catdog',
    data_type='feature',
    data_dir='/media/disk2/Data',
    model='convnext_t',
    minor_ratio = 0.05
)

In [14]:
dm.setup()
dm.prepare_data()

tr {(0, 0): tensor(1465), (0, 1): tensor(151), (1, 0): tensor(71), (1, 1): tensor(3027)}
va {(0, 0): tensor(190), (0, 1): tensor(388), (1, 0): tensor(194), (1, 1): tensor(411)}
te {(0, 0): tensor(480), (0, 1): tensor(998), (1, 0): tensor(480), (1, 1): tensor(998)}


In [ ]:
dm.train_dataset.y

In [ ]:
dm.train_dataset.g

In [8]:
import pandas as pd

# y and g
df = pd.DataFrame({
    'y': dm.train_dataset.y,
    'g': dm.train_dataset.g
})


In [ ]:
df.groupby(['y', 'g']).size()

In [ ]:
dm = DataModule(
    dataset='celeba_collar',
    data_type='raw',
    data_dir='/media/disk1/Data',
    module_name='FeatureGenerator'
)
dm.setup()
dm.prepare_data()

In [ ]:
dm.train_dataset.metadata

In [ ]:
dm.train_dataset.metadata.groupby(['targets', 'blonde_collar']).size()

In [ ]:
# loop over training dataloader
for batch in dm.train_dataloader():
    x, y, g, _ = batch
    break

In [ ]:
g

In [9]:
path = '/media/disk2/Data/Features/celeba_collar/convnext_t/tr'
import torch
dat1 = torch.load(f'{path}/0.pt')
dat2 = torch.load(f'{path}/1.pt')
meta = torch.load(f'{path}/metadata.pt')
df = pd.DataFrame(meta).transpose()
df.columns = ['targets', 'blonde_collar', 'idx']

df['targets'] = df['targets'].astype(int)
df['blonde_collar'] = df['blonde_collar'].astype(int)
df.groupby(['targets', 'blonde_collar']).size()

targets  blonde_collar
0        0                24235
1        0                 2395
         1                   16
dtype: int64

In [10]:
path = '/media/disk2/Data/Features/catdog/convnext_t/tr'
import torch
dat1 = torch.load(f'{path}/0.pt')
dat2 = torch.load(f'{path}/1.pt')
meta = torch.load(f'{path}/metadata.pt')
df = pd.DataFrame(meta).transpose()
df.columns = ['targets', 'blonde_collar', 'idx']

df['targets'] = df['targets'].astype(int)
df['blonde_collar'] = df['blonde_collar'].astype(int)
df.groupby(['targets', 'blonde_collar']).size()

targets  blonde_collar
0        0                1465
         1                  71
1        0                 151
         1                3027
dtype: int64